# Step 6 · Human review loop

Rate answers to unseen test questions, write corrections, turn them into the next SFT dataset.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
SAVE = False   # set True to write new versions into your lab (they appear in the dashboard)

In [ ]:
from lab import engines
MODEL = latest("rlhf") or latest("rl") or latest("sft") or BASE
test = data.active_test_rows(data.load(latest_ds("test")))
answers = engines.get(registry.model(MODEL)["base_model"]).generate(MODEL, [t["prompt"] for t in test], max_new_tokens=96)
pd.DataFrame({"question": [t["prompt"] for t in test], "answer": answers})

In [ ]:
# A review = rating + optional correction (the correction becomes a training example)
t, a = test[0], answers[0]
review = dict(model_id=MODEL, test_ds=latest_ds("test"), question_id=t["id"], question=t["prompt"], answer=a,
              rating="partly correct", correction="Check warehouse stock and delivery availability before confirming.")
if SAVE:
    registry.add_review(**review, approved=0)
review

In [ ]:
# Corrections → new SFT version; the used test question is retired (it is no longer unseen)
if SAVE:
    base = latest_ds("sft_train")
    new_sft = data.save_version("sft_train", data.load(base) + [{"id": f"corr-{t['id']}", "prompt": t["prompt"],
                                "completion": review["correction"]}], parent_id=base, source="corrections")
    trows = [dict(x, retired=True) if x["id"] == t["id"] else x for x in data.load(latest_ds("test"))]
    print(new_sft, data.save_version("test", trows, parent_id=latest_ds("test"), source="retire"))